[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_10_reinforce_CartPole.ipynb)

<div style="text-align:center">
    <h1>
        REINFORCE
    </h1>
</div>

<br><br>

<div style="text-align:center">
In this notebook we are going to implement the Monte Carlo version of Policy Gradient methods. The REINFORCE algorithm uses the full return to update the policy:
</div>

\begin{equation}
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+1} + \cdots + \gamma^{T-t-1} R_{T}
\end{equation}


<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_policy_network, plot_stats, seed_everything, plot_action_probs


## Import the necessary software libraries:

In [ ]:
import os
import torch
import gym
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch import nn as nn
from torch.optim import AdamW

## Create and preprocess the environment

### Create the environment

In [ ]:
env = gym.make('CartPole-v0')

In [ ]:
dims = env.observation_space.shape[0]
actions = env.action_space.n

print(f"State dimensions: {dims}. Actions: {actions}")
print(f"Sample state: {env.reset()}")

In [ ]:
plt.imshow(env.render(mode='rgb_array'))

### Prepare the environment to work with PyTorch

In [ ]:
class PreprocessEnv(gym.Wrapper):

    def __init__(self, env):
        gym.Wrapper.__init__(self, env)

    def reset(self):
        state = self.env.reset()
        return torch.from_numpy(state).float()

    def step(self, actions):
        actions = actions.squeeze().numpy()
        next_state, reward, done, info = self.env.step(actions)
        next_state = torch.from_numpy(next_state).float()
        reward = torch.tensor(reward).unsqueeze(1).float()
        done = torch.tensor(done).unsqueeze(1)
        return next_state, reward, done, info

In [ ]:
num_envs = os.cpu_count()
parallel_env = gym.vector.make('CartPole-v1', num_envs=num_envs)
seed_everything(parallel_env)
parallel_env = PreprocessEnv(parallel_env)

In [ ]:
parallel_env.reset()

### Create the policy $\pi(s)$

**Your turn.** Build the policy network in the cell below.

In policy-gradient methods we learn the **policy directly**: the network takes a state and outputs a **probability for each action**. A few hidden layers with non-linear activations, then a final layer with one output per action, and crucially a **Softmax** so the outputs form a valid probability distribution that sums to 1.

`nn.Sequential`, `nn.Linear`, `nn.ReLU`, and `nn.Softmax(dim=-1)` are what you need.

### Plot action probabilities

In [ ]:
neutral_state = torch.zeros(4)
left_danger = torch.tensor([-2.3, 0., 0., 0.])
right_danger = torch.tensor([2.3, 0., 0., 0.])

#### Plot a neutral environment

In [ ]:
probs = policy(neutral_state).detach().numpy()
plot_action_probs(probs=probs, labels=['Move Left', 'Move Right'])

#### Plot a state where the cart is too far left

In [ ]:
probs = policy(left_danger).detach().numpy()
plot_action_probs(probs=probs, labels=['Move Left', 'Move Right'])

#### Plot a state where the cart is too far right

In [ ]:
probs = policy(right_danger).detach().numpy()
plot_action_probs(probs=probs, labels=['Left', 'Right'])

## Implement the algorithm

### REINFORCE -- the idea you are about to code

REINFORCE is the simplest policy-gradient method:

1. **Play a full episode** by sampling actions from the current policy.
2. For each step compute the **return** $G_t$ -- the total discounted reward from that step to the end of the episode.
3. **Update the policy** so actions followed by a high return become more likely, and actions followed by a low return become less likely.

The key tool is the **log-probability trick**: nudge $\log \pi(a_t \mid s_t)$ in proportion to $G_t$.

**Your turn.** Write the `reinforce` training loop in the cell below. A checklist:
- Create an optimiser for the policy.
- For each episode: roll out a full trajectory, **storing** each `(state, action, reward)`.
- Then walk **backwards** through the trajectory accumulating the discounted return $G = r_t + \gamma G$.
- At each step compute the log-probability of the action taken and form the policy-gradient loss (optionally add a small **entropy** bonus to keep exploring).
- Take a gradient step.

Store the returned statistics so you can plot them.

**Finally:** call your `reinforce` function to train the policy and save the result in a variable named `stats` so the plotting cell below works.

## Show results

### Show execution stats

In [ ]:
plot_stats(stats)

### Plot action probabilities

#### Plot a neutral environment

In [ ]:
probs = policy(neutral_state).detach().numpy()
plot_action_probs(probs=probs, labels=['Move Left', 'Move Right'])

#### Plot a state where the cart is too far left

In [ ]:
probs = policy(left_danger).detach().numpy()
plot_action_probs(probs=probs, labels=['Move Left', 'Move Right'])

#### Plot a state where the cart is too far right

In [ ]:
probs = policy(right_danger).detach().numpy()
plot_action_probs(probs=probs, labels=['Left', 'Right'])

### Test the resulting agent

In [ ]:
test_policy_network(env, policy, episodes=5)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch.13](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)